In [42]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder, PowerTransformer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer # т.н. преобразователь колонок
from sklearn.linear_model import SGDRegressor, LinearRegression
from sklearn.metrics import root_mean_squared_error
import numpy as np
import matplotlib.pyplot as plt
import pickle
import mlflow
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from mlflow.models import infer_signature
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeRegressor

mlflow.set_experiment("ML_lab3")

<Experiment: artifact_location='/home/katerine/mlruns/3', creation_time=1776279549498, experiment_id='3', last_update_time=1776279549498, lifecycle_stage='active', name='ML_lab3', tags={}, workspace='default'>

In [43]:
def preprocessing_data_frame(frame):
    df = frame.copy()
    cat_columns = ['gender', 'race/ethnicity', 'parental level of education', 'lunch', 'test preparation course']
    num_columns = ['math score', 'reading score', 'writing score']
    
    question_score = df[(df['math score'] < 0) | (df['math score'] > 100)]
    df = df.drop(question_score.index)
    question_score = df[(df['reading score'] < 0) | (df['reading score'] > 100)]
    df = df.drop(question_score.index)
    question_score = df[(df['writing score'] < 0) | (df['writing score'] > 100)]
    df = df.drop(question_score.index)

    question_gender = df[~((df['gender'] == 'male') | (df['gender'] == 'female'))]
    df = df.drop(question_gender.index)
    
    df = df.reset_index(drop=True)  
    ordinal = OrdinalEncoder()
    ordinal.fit(df[cat_columns]);
    Ordinal_encoded = ordinal.transform(df[cat_columns])
    df_ordinal = pd.DataFrame(Ordinal_encoded, columns=cat_columns)
    df[cat_columns] = df_ordinal[cat_columns]
    return df

def scale_frame(frame):
    df = frame.copy()
    X,y = df.drop(columns = ['math score']), df['math score']
    scaler = StandardScaler()
    power_trans = PowerTransformer()
    X_scale = scaler.fit_transform(X.values)
    Y_scale = power_trans.fit_transform(y.values.reshape(-1,1))
    return X_scale, Y_scale, power_trans

In [44]:
df = pd.read_csv('https://raw.githubusercontent.com/UsumiMin/lab_ML_1/refs/heads/main/StudentsPerformance.csv', delimiter = ',')
df.head()

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


In [45]:
df_proc = preprocessing_data_frame(df)
X,Y, power_trans = scale_frame(df_proc)
# разбиваем на тестовую и валидационную выборки
X_train, X_val, y_train, y_val = train_test_split(X, Y,
                                                  test_size=0.3,
                                                  random_state=42)
df_proc.head()

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,0.0,1.0,1.0,1.0,1.0,72,72,74
1,0.0,2.0,4.0,1.0,0.0,69,90,88
2,0.0,1.0,3.0,1.0,1.0,90,95,93
3,1.0,0.0,0.0,0.0,1.0,47,57,44
4,1.0,2.0,4.0,1.0,1.0,76,78,75


In [46]:
def eval_metrics(actual, pred):
    rmse = np.sqrt(mean_squared_error(actual, pred))
    mae = mean_absolute_error(actual, pred)
    r2 = r2_score(actual, pred)
    return rmse, mae, r2

In [47]:
params = {'alpha': [0.0001, 0.001, 0.01, 0.05, 0.1 ],
      'l1_ratio': [0.001, 0.05, 0.01, 0.2]
 }
with mlflow.start_run(run_name="SGD_Regressor"):
    lr = SGDRegressor(random_state=42)
    clf = GridSearchCV(lr, params, cv = 5)
    clf.fit(X_train, y_train.reshape(-1))
    best = clf.best_estimator_
    y_pred = best.predict(X_val)
    y_price_pred = power_trans.inverse_transform(y_pred.reshape(-1,1))
    (rmse, mae, r2)  = eval_metrics(power_trans.inverse_transform(y_val), y_price_pred)
    alpha = best.alpha
    l1_ratio = best.l1_ratio
    mlflow.log_param("alpha", alpha)
    mlflow.log_param("l1_ratio", l1_ratio)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2", r2)
    mlflow.log_metric("mae", mae)
    predictions = best.predict(X_train)
    signature = infer_signature(X_train, predictions)
    mlflow.sklearn.log_model(best, "model", signature=signature)
    
with mlflow.start_run(run_name="LinearRegression"):
    lr = LinearRegression()
    lr.fit(X_train, y_train.reshape(-1))
    y_pred = lr.predict(X_val)
    y_price_pred = power_trans.inverse_transform(y_pred.reshape(-1,1))
    (rmse, mae, r2)  = eval_metrics(power_trans.inverse_transform(y_val), y_price_pred)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2", r2)
    mlflow.log_metric("mae", mae)
    predictions = lr.predict(X_train)
    signature = infer_signature(X_train, predictions)
    mlflow.sklearn.log_model(lr, "model", signature=signature)

params = {'max_depth': [3, 5, 10, None], 'min_samples_split': [2, 5]}
with mlflow.start_run(run_name="Decision_Tree"):
    lr = DecisionTreeRegressor(random_state=42)
    clf = GridSearchCV(lr, params, cv = 5)
    clf.fit(X_train, y_train.reshape(-1))
    best = clf.best_estimator_
    y_pred = best.predict(X_val)
    y_price_pred = power_trans.inverse_transform(y_pred.reshape(-1,1))
    (rmse, mae, r2)  = eval_metrics(power_trans.inverse_transform(y_val), y_price_pred)
    max_depth = best.max_depth
    min_samples_split = best.min_samples_split
    mlflow.log_param("max_depth", max_depth)
    mlflow.log_param("min_samples_split", min_samples_split)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2", r2)
    mlflow.log_metric("mae", mae)
    
    predictions = best.predict(X_train)
    signature = infer_signature(X_train, predictions)
    mlflow.sklearn.log_model(best, "model", signature=signature)

2026/04/16 01:15:36 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/16 01:15:36 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/16 01:15:41 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/16 01:15:41 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_p

In [48]:
best.predict([X_val[22]]).reshape(-1,1)

array([[-0.35187082]])

In [49]:
power_trans.inverse_transform(best.predict([X_val[22]]).reshape(-1,1))

array([[61.35461946]])

In [50]:
!ls ./mlruns/3/models

m-155b05ecb7e54ddbb06c96ef18d2a187  m-a71a8e6ef06f4235a9f96513187e1f5d
m-28e90ef22d0e49ecb0e9039ca9c35a24  m-b16bad9a4b1a4c07b2d5e4f3e633733a
m-3b32f21bb09744b192cc18e56e76823b  m-b3c003df000d420e88df004f9c015f9d
m-3ef8adf8face40778934d8c8651787b7  m-db334bfc4c4f46e6a4a8d5874629aee1
m-7949d320e5b645d1939edba57dbebdb9  m-e218eab1a9914168bf57cd27cd4b78c7
m-90df4d1e10d84eaf97abba8467fee54f  m-f21997f5266e46f496c0c2d570bc6546


In [51]:
## История запуска моделей

In [52]:
dfruns = mlflow.search_runs()
frame = dfruns.sort_values("metrics.r2", ascending=False)
frame

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.r2,metrics.rmse,metrics.mae,params.max_depth,params.min_samples_split,params.l1_ratio,params.alpha,tags.mlflow.source.type,tags.mlflow.source.name,tags.mlflow.user,tags.mlflow.runName
0,02992988b37b4e2393646636ccaf00e9,3,FINISHED,/home/katerine/mlruns/3/02992988b37b4e23936466...,2026-04-15 20:15:43.270000+00:00,2026-04-15 20:15:45.585000+00:00,0.822186,6.651466,5.262593,5,5,None,None,NOTEBOOK,mlflow.ipynb,katerine,Decision_Tree
3,6ce1139703784f7a9adef2c1eda05baf,3,FINISHED,/home/katerine/mlruns/3/6ce1139703784f7a9adef2...,2026-04-15 20:01:22.196000+00:00,2026-04-15 20:01:24.426000+00:00,0.822186,6.651466,5.262593,5,5,None,None,NOTEBOOK,mlflow.ipynb,katerine,Decision_Tree
2,261c2fba39704509a41a4210a4e92a9a,3,FINISHED,/home/katerine/mlruns/3/261c2fba39704509a41a42...,2026-04-15 20:15:36.677000+00:00,2026-04-15 20:15:40.980000+00:00,0.758535,7.751066,4.855758,None,None,0.001,0.0001,NOTEBOOK,mlflow.ipynb,katerine,SGD_Regressor
5,8c2776a366534819bd0fb09b9083b68a,3,FINISHED,/home/katerine/mlruns/3/8c2776a366534819bd0fb0...,2026-04-15 20:01:17.241000+00:00,2026-04-15 20:01:20.253000+00:00,0.758535,7.751066,4.855758,None,None,0.001,0.0001,NOTEBOOK,mlflow.ipynb,katerine,SGD_Regressor
13,51c069d8e029467ebd7c15450c6aa5d7,3,FINISHED,/home/katerine/mlruns/3/51c069d8e029467ebd7c15...,2026-04-15 19:38:02.698000+00:00,2026-04-15 19:38:06.518000+00:00,0.758535,7.751066,4.855758,None,None,0.001,0.0001,NOTEBOOK,mlflow.ipynb,katerine,rebellious-elk-453
8,5095b58c289646e7b9d533900e7ce2d2,3,FINISHED,/home/katerine/mlruns/3/5095b58c289646e7b9d533...,2026-04-15 20:00:39.208000+00:00,2026-04-15 20:00:41.807000+00:00,0.758535,7.751066,4.855758,None,None,0.001,0.0001,NOTEBOOK,mlflow.ipynb,katerine,SGD_Regressor
12,df7c2dcafba94db2930f715affb9734e,3,FINISHED,/home/katerine/mlruns/3/df7c2dcafba94db2930f71...,2026-04-15 19:57:37.582000+00:00,2026-04-15 19:57:39.941000+00:00,0.758535,7.751066,4.855758,None,None,0.001,0.0001,NOTEBOOK,mlflow.ipynb,katerine,SGD_Regressor
10,9e43aedbd0ab494db80297b0110aa021,3,FINISHED,/home/katerine/mlruns/3/9e43aedbd0ab494db80297...,2026-04-15 19:59:38.729000+00:00,2026-04-15 19:59:41.953000+00:00,0.758535,7.751066,4.855758,None,None,0.001,0.0001,NOTEBOOK,mlflow.ipynb,katerine,SGD_Regressor
15,34e44c60419f4778a6c80d6d46da6b3b,3,FINISHED,/home/katerine/mlruns/3/34e44c60419f4778a6c80d...,2026-04-15 18:59:21.775000+00:00,2026-04-15 18:59:24.064000+00:00,0.758535,7.751066,4.855758,None,None,0.001,0.0001,NOTEBOOK,mlflow.ipynb,katerine,skillful-panda-11
1,440f998a06c24ecb8b8ed7711de370ab,3,FINISHED,/home/katerine/mlruns/3/440f998a06c24ecb8b8ed7...,2026-04-15 20:15:40.986000+00:00,2026-04-15 20:15:43.262000+00:00,0.726660,8.246807,4.889435,None,None,None,None,NOTEBOOK,mlflow.ipynb,katerine,LinearRegression


In [53]:
model_uri = dfruns.sort_values("metrics.r2", ascending=False).iloc[0]['artifact_uri']
path2model = dfruns.sort_values("metrics.r2", ascending=False).iloc[0]['artifact_uri'].replace("file://","") + '/model' #путь до эксперимента с лучшей моделью

best_model = mlflow.search_logged_models(
    experiment_ids=["3"],
    max_results=1,
    order_by=[{"field_name": "metrics.accuracy", "ascending": False}],
    output_format="list",
)[0]
print(best_model)
path2model = f"models:/{best_model.model_id}"
print(path2model)

LoggedModel(artifact_location='/home/katerine/mlruns/3/models/m-90df4d1e10d84eaf97abba8467fee54f/artifacts', creation_timestamp=1776284143402, experiment_id='3', last_updated_timestamp=1776284145574, model_id='m-90df4d1e10d84eaf97abba8467fee54f', model_type=None, model_uri='models:/m-90df4d1e10d84eaf97abba8467fee54f', name='model', source_run_id='02992988b37b4e2393646636ccaf00e9', status=<LoggedModelStatus.READY: 'READY'>, status_message=None)
models:/m-90df4d1e10d84eaf97abba8467fee54f


In [54]:
#loaded_model = mlflow.pyfunc.load_model(path2model)
loaded_model = mlflow.sklearn.load_model(path2model)
loaded_model

,"criterion criterion: {""squared_error"", ""friedman_mse"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in the half mean Poisson deviance to find splits... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 0.24 Poisson deviance criterion.",'squared_error'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.For an example of how ``max_depth`` influences the model, see:ref:`sphx_glr_auto_examples_tree_plot_tree_regression.py`.",5
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",5
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",42
,"max_leaf

In [55]:
test_input = np.array([[
    0.0,
    1.0,
    1.0,
    1.0,
    72,
    72,
    74
]])

prediction = loaded_model.predict(test_input)
prediction

array([1.75246205])

### Запускает модель в сервис локально 

In [56]:
!mlflow models serve -m /home/katerine/mlruns/3/models/m-90df4d1e10d84eaf97abba8467fee54f/artifacts -p 5001 --env-manager local

2026/04/16 01:17:24 INFO mlflow.models.flavor_backend_registry: Selected backend for flavor 'python_function'
2026/04/16 01:17:24 INFO mlflow.pyfunc.backend: === Running command 'exec uvicorn --host 127.0.0.1 --port 5001 --workers 1 mlflow.pyfunc.scoring_server.app:app'
INFO:     Started server process [248933]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:5001 (Press CTRL+C to quit)
INFO:     127.0.0.1:52648 - "POST /invocations HTTP/1.1" 200 OK
^C
INFO:     Shutting down


### Тестовый запрос к модели через curl

In [ ]:
!curl http://127.0.0.1:5001/invocations -H "Content-Type:application/json"  --data '{"inputs": [[0.0, 1.0, 1.0, 1.0, 72, 72, 74]]}'